In [1]:
import pandas as pd
import numpy as np
import time
import os
from sklearn.preprocessing import MinMaxScaler

import matplotlib.pyplot as plt
import seaborn as sns
import yaml

with open("../config.yaml", "r") as file:
        config = yaml.safe_load(file)

output_dir = "../data/clean/"
config

{'input_data': {'file1': '../data/raw/2024-10-01_performance_mobile_tiles.parquet',
  'file2': '../data/raw/ADE_4-0_GPKG_WGS84G_FRA-ED2026-01-19.gpkg',
  'file3': '../data/raw/fichier_diffusion_2024.xlsx',
  'file4': '../data/raw/2024-10-01_performance_fixed_tiles.parquet'},
 'output_data': {'file1': '../data/clean/perf_admin_fr_df.csv',
  'file2': '../data/clean/file2_cleaned.csv'}}

In [2]:
df = pd.read_csv(config['output_data']['file1'], dtype={
        'code_insee_dept': str,
        'code_insee_comm': str,
        'zip_code': str
    })
df.head()

,avg_down_mbps,avg_up_mbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,nbr_tests,nbr_devices,region_name,code_insee_region,code_siren_region,...,code_insee_dept,commune_name,commune_status,code_insee_comm,comm_population,zip_code,superficie_cadastrale,quadkey,geometry,LIBDENS
0,325.78,330.40,14,63.0,1496.0,1,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,313133210211310,"POLYGON ((-1.9390869140625 49.70672028103242, ...",Rural
1,318.05,462.16,9,82.0,177.0,1,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,313133210211333,"POLYGON ((-1.93359375 49.69606181911564, -1.93...",Rural
2,140.71,86.54,14,167.0,178.0,2,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,313133210300032,"POLYGON ((-1.9171142578125 49.71027258210569, ...",Rural
3,395.56,291.00,17,337.0,480.0,6,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,313133210300033,"POLYGON ((-1.91162109375 49.71027258210569, -1...",Rural
4,237.31,257.19,10,337.0,145.0,2,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,313133210300132,"POLYGON ((-1.8951416015625 49.71027258210569, ...",Rural


In [3]:
#Fill u p enmpty LIBDENS based on the comm_population col
def fill_up_empty_libdens(df):
    df1 = df.copy()

    mask = df1['LIBDENS'].isnull()
    df1.loc[mask & (df1['comm_population'] <= 2000), 'LIBDENS'] = "Rural"
    df1.loc[mask & ((df1['comm_population'] > 2000)
                   & (df1['comm_population'] <= 10000)), 'LIBDENS'] = "Intermediate urban"
    df1.loc[mask & (df1['comm_population'] > 10000), 'LIBDENS'] ="Dense urban"

    # translate 'comm_population' col to English
    df1['LIBDENS'] = df1['LIBDENS'].replace({'Urbain intermédiaire': "Intermediate urban", "Urbain dense":"Dense urban"})
    
   
    return df1
    
df1_urban = fill_up_empty_libdens(df)
df1_urban.head()

,avg_down_mbps,avg_up_mbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,nbr_tests,nbr_devices,region_name,code_insee_region,code_siren_region,...,code_insee_dept,commune_name,commune_status,code_insee_comm,comm_population,zip_code,superficie_cadastrale,quadkey,geometry,LIBDENS
0,325.78,330.40,14,63.0,1496.0,1,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,313133210211310,"POLYGON ((-1.9390869140625 49.70672028103242, ...",Rural
1,318.05,462.16,9,82.0,177.0,1,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,313133210211333,"POLYGON ((-1.93359375 49.69606181911564, -1.93...",Rural
2,140.71,86.54,14,167.0,178.0,2,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,313133210300032,"POLYGON ((-1.9171142578125 49.71027258210569, ...",Rural
3,395.56,291.00,17,337.0,480.0,6,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,313133210300033,"POLYGON ((-1.91162109375 49.71027258210569, -1...",Rural
4,237.31,257.19,10,337.0,145.0,2,1,NORMANDIE,28,200053403,...,50,LA HAGUE,Commune simple,50041,11484,50440,14870,313133210300132,"POLYGON ((-1.8951416015625 49.71027258210569, ...",Rural


In [4]:
# Create latency categories
def latency_cat(df):
    df1 = fill_up_empty_libdens(df)
    df1 = df1.dropna(subset = ['avg_lat_down_ms', 'avg_lat_up_ms'])
    df1 = df1.drop(columns = 'zip_code')
    bins = [0, 20, 50, 100, 150, float('inf')]
    labels = ["Excellent", "Good", "Acceptable", "Fair", "Poor"]
    df1['load_lat_ms'] = (df1['avg_lat_down_ms'] + df1['avg_lat_up_ms'])/2
    df1["latency_category"] = pd.cut(
                                        df1['load_lat_ms'],
                                        bins = bins,
                                        labels = labels,
                                        right = False
                                      )
  
    return df1

df1 = latency_cat(df)
df1.head()

,avg_down_mbps,avg_up_mbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,nbr_tests,nbr_devices,region_name,code_insee_region,code_siren_region,...,commune_name,commune_status,code_insee_comm,comm_population,superficie_cadastrale,quadkey,geometry,LIBDENS,load_lat_ms,latency_category
0,325.78,330.40,14,63.0,1496.0,1,1,NORMANDIE,28,200053403,...,LA HAGUE,Commune simple,50041,11484,14870,313133210211310,"POLYGON ((-1.9390869140625 49.70672028103242, ...",Rural,779.5,Poor
1,318.05,462.16,9,82.0,177.0,1,1,NORMANDIE,28,200053403,...,LA HAGUE,Commune simple,50041,11484,14870,313133210211333,"POLYGON ((-1.93359375 49.69606181911564, -1.93...",Rural,129.5,Fair
2,140.71,86.54,14,167.0,178.0,2,1,NORMANDIE,28,200053403,...,LA HAGUE,Commune simple,50041,11484,14870,313133210300032,"POLYGON ((-1.9171142578125 49.71027258210569, ...",Rural,172.5,Poor
3,395.56,291.00,17,337.0,480.0,6,1,NORMANDIE,28,200053403,...,LA HAGUE,Commune simple,50041,11484,14870,313133210300033,"POLYGON ((-1.91162109375 49.71027258210569, -1...",Rural,408.5,Poor
4,237.31,257.19,10,337.0,145.0,2,1,NORMANDIE,28,200053403,...,LA HAGUE,Commune simple,50041,11484,14870,313133210300132,"POLYGON ((-1.8951416015625 49.71027258210569, ...",Rural,241.0,Poor


In [5]:
df1= fill_up_empty_libdens(df)
df1.shape # (229007, 21)
#df1.isna().sum()

(230321, 21)

In [13]:
#Broaddband quality score
def broadband_qty_score(df):
    df1 = latency_cat(df)
    #Normalize the load_lat_ms, avg_up_mbps and 	avg_down_mbps
    #create an instance oft he normalizer
    normalizer = MinMaxScaler()

    #fit the normalizer
    df1['down_speed_norm'] = normalizer.fit_transform(df1[['avg_down_mbps']])
    df1['up_speed_norm'] = normalizer.fit_transform(df1[['avg_up_mbps']])
    df1['load_lat_norm'] = 1 - normalizer.fit_transform(df1[['load_lat_ms']])
    df1.loc[df1['department_name'] == 'LOT', 'region_name'] = "OCCITANIE"
    df1.loc[df1['department_name'] == 'LOT', 'code_insee_region'] = 76

    #Calculate the borandband score
    df1['broadband_score'] = round((df1['down_speed_norm']*0.5 + df1['up_speed_norm']*0.2 + df1['load_lat_norm']*0.3)*100, 2)

    return df1

df1 = broadband_qty_score(df)
df1.to_csv(f"{output_dir}fixed_internet_perf_clean.csv", index=False, encoding= "utf-8", sep = ",")
df1.head()
df1.info()
  

<class 'pandas.DataFrame'>
Index: 229007 entries, 0 to 230320
Data columns (total 26 columns):
 #   Column                 Non-Null Count   Dtype   
---  ------                 --------------   -----   
 0   avg_down_mbps          229007 non-null  float64 
 1   avg_up_mbps            229007 non-null  float64 
 2   avg_lat_ms             229007 non-null  int64   
 3   avg_lat_down_ms        229007 non-null  float64 
 4   avg_lat_up_ms          229007 non-null  float64 
 5   nbr_tests              229007 non-null  int64   
 6   nbr_devices            229007 non-null  int64   
 7   region_name            229007 non-null  str     
 8   code_insee_region      229007 non-null  int64   
 9   code_siren_region      229007 non-null  int64   
 10  department_name        229007 non-null  str     
 11  code_insee_dept        229007 non-null  str     
 12  commune_name           229007 non-null  str     
 13  commune_status         229007 non-null  str     
 14  code_insee_comm        229007 non-nu

In [114]:
# defining fact and dimensions tables to use in BigQuery
def create_performance_tables(df):
    """
    create the fact table
    """
    df1 = broadband_qty_score(df)
    #create prefromance ID
    len_perf_tbl = len(df1)
    perf_id = [i for i in range(1, len_perf_tbl +1)]
        
    #Create performace table: with normalized data
    performance_table = df1[['avg_down_mbps',
                              'avg_up_mbps',
                              'avg_lat_ms',
                              'avg_lat_down_ms',
                              'avg_lat_up_ms',
                              'load_lat_ms',
                              'latency_category',
                              'nbr_tests',
                              'nbr_devices',
                              'down_speed_norm',
                              'up_speed_norm',
                              'load_lat_norm',
                              'broadband_score',
                              'code_insee_region',
                              'code_insee_dept',
                              'code_insee_comm'
                             'quadkey',
                             'geometry'
                             ]]
    performance_table.insert(0, 'perf_id', perf_id)
    
   #cretae region table
    region_table = df1[['code_insee_region', 
                    'region_name', 
                    'code_siren_region'
                   ]
                   ]
    region_table = region_table.drop_duplicates(subset = 'code_insee_region')
    
    #cretae department table
    department_table = df1[['code_insee_dept',
                         'department_name',
                         'code_insee_region'
                        ]
                        ] 
    department_table = department_table.drop_duplicates(subset = 'code_insee_dept')

    #cretae department table
    commune_table = df1[['code_insee_comm',
                      'commune_name',
                      'commune_status',
                      'comm_population',
                      'superficie_cadastrale',
                      'LIBDENS',
                      'code_insee_dept'
                     ]]
    commune_table = commune_table.drop_duplicates(subset = 'code_insee_comm')
    commune_table = commune_table.reset_index(drop = True)
    
    return performance_table, region_table, department_table, commune_table               
                         
performance_table, region_table, department_table, commune_table = create_performance_tables(df)

performance_table.to_csv(f"{output_dir}performance_table.csv", index=False, encoding= "utf-8", sep = ",")
region_table.to_csv(f"{output_dir}region_table.csv", index=False, encoding= "utf-8", sep = ",")
department_table.to_csv(f"{output_dir}department_table.csv", index=False, encoding= "utf-8", sep = ",")
commune_table.to_csv(f"{output_dir}commune_table.csv", index=False, encoding= "utf-8", sep = ",")



In [115]:
performance_table.info()


In [108]:
performance_table.shape